# classification ceu1

In [ ]:
# Read the dataset
df = pd.read_csv("/home/sk/Projects/Astrophysics/data/ATLAS.csv")
print("Original dataset shape:", df.shape)

# Define features and target
features = ["Sp2","flux_ap2_36","flux_ap2_45","flux_ap2_58","flux_ap2_80","MAG_APER_4_G","MAG_APER_4_R","MAG_APER_4_I","MAG_APER_4_Z"]
target = "z"

# Convert data to NumPy arrays
X = df[features].to_numpy(dtype=np.float32)
y = df[target].to_numpy(dtype=np.float32)

# Create 15 bins using the exact GitHub function
num_bins = 15
y_binned, bin_edges, bin_values = binDataFunc(y.copy(), num_bins)

# print("Number of classes:", num_bins)
# print("Class counts:", [np.sum(y_binned == z) for z in bin_values])
# print("Class redshifts:", bin_values)

# Create the exact 70/30 train-test split
X_train, X_test, y_train, y_test = randomsplit(X, y_binned)

print("Training sources:", len(X_train))
print("Testing sources:", len(X_test))

# Define eta_0.15 scorer
def eta_015(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    residuals = (y_true - y_pred) / (1 + y_true)
    return np.mean(np.abs(residuals) > 0.15)

eta015_scorer = make_scorer(eta_015, greater_is_better=False)

# Define 10-fold cross-validation
kfold = KFold(n_splits=10, random_state=10, shuffle=True)

# Define Euclidean kNN pipeline
model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(p=2))
])

# Search k = 2 to 20 using GridSearchCV
grid = GridSearchCV(
    estimator=model,
    param_grid={"knn__n_neighbors": range(2, 21)},
    scoring={"eta015": eta015_scorer, "accuracy": "accuracy"},
    refit="eta015",
    cv=kfold,
    n_jobs=-1
)

grid.fit(X_train, y_train.astype(str))

# Display cross-validation results
# cv_results = grid.cv_results_
# for k, eta, acc in zip(cv_results["param_knn__n_neighbors"], -cv_results["mean_test_eta015"], cv_results["mean_test_accuracy"]):
#     print(f"k={int(k):2d} | eta_0.15={100*eta:.4f}% | accuracy={acc:.4f}")

# Best k selected using minimum eta_0.15
best_k = grid.best_params_["knn__n_neighbors"]
best_cv_eta = -grid.best_score_

# print("\nBest k:", best_k)
# print("Best CV eta_0.15:", 100 * best_cv_eta, "%")

# Final model is already refitted by GridSearchCV
final_model = grid.best_estimator_
final_prediction = final_model.predict(X_test).astype(np.float32)

# Calculate residuals
residuals = (y_test - final_prediction) / (1 + y_test)

# Calculate final metrics
eta_015_final = np.mean(np.abs(residuals) > 0.15)
sigma = np.std(residuals)
eta_2sigma = np.mean(np.abs(residuals) > 2 * sigma)
accuracy = np.mean(final_prediction == y_test)

print("\nCEu1 RESULTS")
print("k:", best_k)
print("eta_0.15 (%):", 100 * eta_015_final)
print("eta_2sigma (%):", 100 * eta_2sigma)
print("Accuracy:", accuracy)
print("sigma:", sigma)

# Ceu2

In [ ]:
# Read the datasets
df_elais = pd.read_csv("/home/sk/Projects/Astrophysics/data/ELAIS-S1.csv")
df_cdfs = pd.read_csv("/home/sk/Projects/Astrophysics/data/CDFS.csv")
print("Original dataset shape:",df.shape)

# Define features and target
features = ["Sp2","flux_ap2_36","flux_ap2_45","flux_ap2_58","flux_ap2_80","MAG_APER_4_G","MAG_APER_4_R","MAG_APER_4_I","MAG_APER_4_Z"]
target = "z"

# Extract features and redshifts
X_train = df_elais[features].to_numpy(dtype=np.float32)
X_test = df_cdfs[features].to_numpy(dtype=np.float32)
z_train = df_elais[target].to_numpy(dtype=np.float32)
z_test = df_cdfs[target].to_numpy(dtype=np.float32)

# Combine redshifts and create 15 bins
z_all = np.concatenate([z_train,z_test])
z_all_binned,bin_edges,bin_values = binDataFunc(z_all.copy(),15)

# Separate binned redshifts
n_train = len(z_train)
y_train = z_all_binned[:n_train]
y_test = z_all_binned[n_train:]

# print("Number of classes:",15)
# print("Class counts:",[np.sum(z_all_binned == z) for z in bin_values])
# print("Class redshifts:",bin_values)
# print("Training sources:",len(X_train))
# print("Testing sources:",len(X_test))

# Define eta_0.15 scorer
def eta_015(y_true,y_pred):
    y_true = np.asarray(y_true,dtype=np.float32)
    y_pred = np.asarray(y_pred,dtype=np.float32)
    residuals = (y_true-y_pred)/(1+y_true)
    return np.mean(np.abs(residuals)>0.15)

eta015_scorer = make_scorer(eta_015,greater_is_better=False)

# Define 10-fold cross-validation
kfold = KFold(n_splits=10,random_state=10,shuffle=True)

# Define Euclidean kNN pipeline
model = Pipeline([
    ("scaler",StandardScaler()),
    ("knn",KNeighborsClassifier(p=2))
])

# Search k from 2 to 20
grid = GridSearchCV(
    estimator=model,
    param_grid={"knn__n_neighbors":range(2,21)},
    scoring={"eta015":eta015_scorer,"accuracy":"accuracy"},
    refit="eta015",
    cv=kfold,
    n_jobs=-1
)

# Perform cross-validation on ELAIS-S1
grid.fit(X_train,y_train.astype(str))

# Get the best k
best_k = grid.best_params_["knn__n_neighbors"]
best_cv_eta = -grid.best_score_

# print("\nCROSS-VALIDATION")
# print("Best k:",best_k)
# print("Best CV eta_0.15:",100*best_cv_eta,"%")

# Predict the independent CDFS test set
final_model = grid.best_estimator_
final_prediction = final_model.predict(X_test).astype(np.float32)

# Calculate residuals
residuals = (y_test-final_prediction)/(1+y_test)

# Calculate final metrics
eta_015_final = np.mean(np.abs(residuals)>0.15)
sigma = np.std(residuals)
eta_2sigma = np.mean(np.abs(residuals)>2*sigma)
accuracy = np.mean(final_prediction==y_test)

# Print CEu2 results
print("\nCEu2 RESULTS")
print("Train: ELAIS-S1")
print("Test: CDFS")
print("Method: Euclidean kNN")
print("k:",best_k)
print("eta_0.15 (%):",100*eta_015_final)
print("eta_2sigma (%):",100*eta_2sigma)
print("Accuracy:",accuracy)
print("sigma:",sigma)

# Cma1 test

In [ ]:
# Read the dataset
df = pd.read_csv("/home/sk/Projects/Astrophysics/data/ATLAS.csv")
print("Original dataset shape:",df.shape)

# Define features and target
features = ["Sp2","flux_ap2_36","flux_ap2_45","flux_ap2_58","flux_ap2_80","MAG_APER_4_G","MAG_APER_4_R","MAG_APER_4_I","MAG_APER_4_Z"]
target = "z"

# Convert data to NumPy arrays
X = df[features].to_numpy(dtype=np.float32)
y = df[target].to_numpy(dtype=np.float32)

# Create 15 bins using the exact GitHub function
num_bins = 15
y_binned,bin_edges,bin_values = binDataFunc(y.copy(),num_bins)

print("Number of classes:",num_bins)
print("Class counts:",[np.sum(y_binned==z) for z in bin_values])
print("Class redshifts:",bin_values)

# Create the exact 70/30 train-test split
X_train,X_test,y_train,y_test = randomsplit(X,y_binned)

print("Training sources:",len(X_train))
print("Testing sources:",len(X_test))


# Define Mahalanobis kNN
class MahalanobisKNN(BaseEstimator,ClassifierMixin):

    def __init__(self,n_neighbors=5):
        self.n_neighbors = n_neighbors

    def fit(self,X,y):
        self.scaler_ = StandardScaler()
        X_scaled = self.scaler_.fit_transform(X)

        self.covariance_ = np.cov(X_scaled,rowvar=False)

        self.model_ = KNeighborsClassifier(
            n_neighbors=self.n_neighbors,
            metric="mahalanobis",
            metric_params={"V":self.covariance_}
        )

        self.model_.fit(X_scaled,y.astype(str))
        return self

    def predict(self,X):
        X_scaled = self.scaler_.transform(X)
        return self.model_.predict(X_scaled)

    def score(self,X,y):
        X_scaled = self.scaler_.transform(X)
        return self.model_.score(X_scaled,y.astype(str))


# Define eta_0.15 scorer
def eta_015(y_true,y_pred):
    y_true = np.asarray(y_true,dtype=np.float32)
    y_pred = np.asarray(y_pred,dtype=np.float32)
    residuals = (y_true-y_pred)/(1+y_true)
    return np.mean(np.abs(residuals)>0.15)

eta015_scorer = make_scorer(eta_015,greater_is_better=False)

# Define 10-fold cross-validation
kfold = KFold(n_splits=10,random_state=10,shuffle=True)

# Define Mahalanobis kNN model
model = MahalanobisKNN()

# Search k from 2 to 20
grid = GridSearchCV(
    estimator=model,
    param_grid={"n_neighbors":range(2,21)},
    scoring={"eta015":eta015_scorer,"accuracy":"accuracy"},
    refit="eta015",
    cv=kfold,
    n_jobs=-1
)

# Perform cross-validation
grid.fit(X_train,y_train.astype(str))

# Get the best k
best_k = grid.best_params_["n_neighbors"]
best_cv_eta = -grid.best_score_

print("\nCROSS-VALIDATION")
print("Best k:",best_k)
print("Best CV eta_0.15:",100*best_cv_eta,"%")

# Predict the test set
final_model = grid.best_estimator_
final_prediction = final_model.predict(X_test).astype(np.float32)

# Calculate residuals
residuals = (y_test-final_prediction)/(1+y_test)

# Calculate final metrics
eta_015_final = np.mean(np.abs(residuals)>0.15)
sigma = np.std(residuals)
eta_2sigma = np.mean(np.abs(residuals)>2*sigma)
accuracy = np.mean(final_prediction==y_test)

# Print CMa1 results
print("\nCMa1 RESULTS")
print("Train: ATLAS 70%")
print("Test: ATLAS 30%")
print("Method: Mahalanobis kNN")
print("k:",best_k)
print("eta_0.15 (%):",100*eta_015_final)
print("eta_2sigma (%):",100*eta_2sigma)
print("Accuracy:",accuracy)
print("sigma:",sigma)